<a href="https://colab.research.google.com/github/yashashwanidixit/ShrinkOrSink/blob/main/Image_classifier_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.transforms import v2
from torchsummary import summary
import matplotlib.pyplot as plt
import numpy as np

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.transforms import v2
from torchsummary import summary
import matplotlib.pyplot as plt
import numpy as np

In [19]:
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_built() and torch.backends.mps.is_available():
    device = torch.device("mps")
print(device)

cpu


In [20]:
transform_train=transforms.Compose([

      transforms.RandomResizedCrop(96, scale=(0.8, 1.0)),
      transforms.RandomHorizontalFlip(0.5),
      transforms.ColorJitter(brightness=0.2),
      transforms.ToTensor(),
      transforms.Normalize(
          (0.43, 0.42, 0.39),
          (0.27, 0.26, 0.27)
      )
])


transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.43, 0.42, 0.39),
        (0.27, 0.26, 0.27)
    )
])

train_set = torchvision.datasets.STL10(root='./data', split='train', download=True, transform=transform_train)
test_set = torchvision.datasets.STL10(root='./data', split='test', download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=4, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=4,shuffle=False)

classes = ('airplane', 'bird', 'car', 'cat', 'deer', 'dog', 'horse', 'monkey', 'ship', 'truck')

In [21]:
class ConvNeuralNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.conv3 = nn.Conv2d(128, 256, 3, padding=1)
        self.bn3 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(2, stride=2)

        self.fc1 = nn.Linear(256 * 12 * 12, 240)
        self.fc2 = nn.Linear(240, 120)
        self.fc3 = nn.Linear(120, 10)


        self.dropout = nn.Dropout(0.3)

    def forward(self, x):

        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.log_softmax(self.fc3(x), dim=1)
        return x


net = ConvNeuralNet()
net.to(device)

ConvNeuralNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv3): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=36864, out_features=240, bias=True)
  (fc2): Linear(in_features=240, out_features=120, bias=True)
  (fc3): Linear(in_features=120, out_features=10, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)

In [22]:
import torch
import numpy as np
import random

def set_seed(seed_value):
    torch.manual_seed(seed_value)
    np.random.seed(seed_value)
    random.seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
print(f"Manual seed set to {SEED}")

Manual seed set to 42


In [23]:
def view_classification(image, probabilities):
    probabilities = probabilities.data.numpy().squeeze()

    fig, (ax1, ax2) = plt.subplots(figsize=(6,9), ncols=2)


    mean = torch.tensor([0.43, 0.42, 0.39]).view(3, 1, 1)
    std = torch.tensor([0.27, 0.26, 0.27]).view(3, 1, 1)
    denormalized_image = image * std + mean
    denormalized_image = denormalized_image.permute(1, 2, 0)
    denormalized_image = torch.clamp(denormalized_image, 0, 1)

    ax1.imshow(denormalized_image)
    ax1.axis('off')
    ax2.barh(np.arange(10), probabilities)
    ax2.set_aspect(0.1)
    ax2.set_yticks(np.arange(10))
    ax2.set_yticklabels(classes)
    ax2.set_title('Class Probability')
    ax2.set_xlim(0, 1.1)
    plt.tight_layout()

In [24]:
loss_function = nn.NLLLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

epochs = 10
for epoch in range(epochs):

    running_loss = 0.0


    print_frequency = 250
    for i, data in enumerate(train_loader):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = net(inputs)
        loss = loss_function(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % print_frequency == (print_frequency - 1):
            print(f'[{epoch + 1}/{epochs}, {i + 1:5d}] loss: {running_loss / print_frequency:.3f}')
            running_loss = 0.0

print('Finished Training')

[1/10,   250] loss: 3.044
[1/10,   500] loss: 2.225
[1/10,   750] loss: 2.179
[1/10,  1000] loss: 2.098
[1/10,  1250] loss: 2.163
[2/10,   250] loss: 2.116
[2/10,   500] loss: 2.104
[2/10,   750] loss: 2.040
[2/10,  1000] loss: 2.003
[2/10,  1250] loss: 1.962
[3/10,   250] loss: 1.964
[3/10,   500] loss: 1.943
[3/10,   750] loss: 1.949
[3/10,  1000] loss: 1.914
[3/10,  1250] loss: 1.952
[4/10,   250] loss: 1.900
[4/10,   500] loss: 1.865
[4/10,   750] loss: 1.875
[4/10,  1000] loss: 1.864
[4/10,  1250] loss: 1.888
[5/10,   250] loss: 1.785
[5/10,   500] loss: 1.795
[5/10,   750] loss: 1.781
[5/10,  1000] loss: 1.860
[5/10,  1250] loss: 1.790
[6/10,   250] loss: 1.766
[6/10,   500] loss: 1.731
[6/10,   750] loss: 1.770
[6/10,  1000] loss: 1.807
[6/10,  1250] loss: 1.745
[7/10,   250] loss: 1.712
[7/10,   500] loss: 1.720
[7/10,   750] loss: 1.682
[7/10,  1000] loss: 1.761
[7/10,  1250] loss: 1.696
[8/10,   250] loss: 1.707
[8/10,   500] loss: 1.714
[8/10,   750] loss: 1.699
[8/10,  1000

In [26]:
correct = 0
total = 0

for images, labels in train_loader:
    outputs = net(images)

    _, predicted = torch.max(outputs.data, 1)


    total += labels.size(0)


    correct += (predicted == labels).sum().item()

train_accuracy = 100 * correct / total
print(f'Training Accuracy: {train_accuracy:.2f}%')

Training Accuracy: 44.86%


In [27]:
correct = 0
total = 0

with torch.no_grad():
    for data in test_loader:
        images, labels = data[0].to(device), data[1].to(device)

        outputs = net(images)

        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy of the network on the 10000 test images: {100 * correct // total} %')

Accuracy of the network on the 10000 test images: 42 %


In [28]:
param_size = 0
for param in net.parameters():
    param_size += param.nelement() * param.element_size()


size_all_mb = (param_size) / 1024**2
print(f'Model size: {size_all_mb:.3f}MB')

Model size: 35.284MB
